# 迷你 DeepSeek-R1-Zero：GRPO 数学题实验

目标：从已经预训练的 Qwen3 Base 出发，用 GSM8K 的可验证答案奖励进行后训练，观察准确率与生成行为。这里的“Zero”指不先做本实验的 SFT，不代表模型从零学习数学知识。

版本：TRL 1.14.1；按顺序运行各节。默认 80 GB 单卡档位会训练并消耗算力；先用 smoke 跑通接口。Notebook 当前不包含已执行的结果。

教材链接：https://posttrain.gaozhijun.me/docs/lecture-4/lab/ 。教材的一些参数来自旧版 TRL，本文件按锁定版本调整；具体以实际安装版本和训练日志为准。


## 0. 核心原理：GRPO

PPO 类语言模型训练通常使用价值函数网络估计优势；GRPO 用同一题目采出的 G 个回复的组内奖励作为相对基线，省去单独训练的 Critic，但仍需为 G 次生成付出成本。

对题目 q 采样 G 个回复，按标准答案计算 0/1 奖励。实现中会对组内奖励做中心化和缩放，再用带裁剪的策略目标更新模型。组内全对或全错时，正确性奖励没有相对差异，基本不能提供有效更新。此 notebook 额外设 beta=0.001，引入参考模型 KL 约束。

完整数学公式推导与定义如下。

---

### 1. 组内相对优势计算（Group Relative Advantage）

对于给定的输入问题 $q \sim P(Q)$，从当前旧策略 $\pi_{\theta_{\text{old}}}$ 中独立采样 $G$ 个候选回答：


$$\{o_1, o_2, \dots, o_G\} \sim \pi_{\theta_{\text{old}}}(O \mid q)$$

每个回答 $o_i$ 经由标量奖励函数计算得到奖励 $r_i$（如可验证规则输出的 0 或 1）。

计算这 $G$ 个奖励的组内均值 $\bar{r}$ 和组内标准差 $\sigma_r$：


$$\bar{r} = \frac{1}{G} \sum_{j=1}^G r_j$$

$$\sigma_r = \sqrt{\frac{1}{G} \sum_{j=1}^G (r_j - \bar{r})^2}$$

第 $i$ 个回答的**组内相对优势（Group Relative Advantage）** $A_i$ 定义为：


$$A_i = \frac{r_i - \bar{r}}{\sigma_r + \epsilon_{\text{eps}}}$$

> **数值边界（全对/全错情况）：**
> 当采样出的 $G$ 个回答全对（所有 $r_i=1$）或全错（所有 $r_i=0$）时，$\sigma_r = 0$。加入极小常数 $\epsilon_{\text{eps}}$（通常取 $10^{-8}$ 或 $10^{-4}$）后，$A_i \approx 0$。这意味着在无相对差异的样本组中，梯度贡献趋近于零。

---

### 2. Token 粒度概率比率（Probability Ratio）

本质就是重要性采样，但是是token粒度，不是序列级。对于第 $i$ 个回答中的第 $t$ 个 Token $o_{i,t}$，当前策略与采样所用旧策略的概率比率为：


$$r_{i,t}(\theta) = \frac{\pi_\theta(o_{i,t} \mid q, o_{i,<t})}{\pi_{\theta_{\text{old}}}(o_{i,t} \mid q, o_{i,<t})}$$

完整序列 $o = (o_1, o_2, \dots, o_T)$ 的严谨重要性采样权重应该是所有 Token 概率比率的累乘：$$W_{\text{seq}}(\theta) = \frac{\pi_\theta(o \mid q)}{\pi_{\theta_{\text{old}}}(o \mid q)} = \prod_{t=1}^T \frac{\pi_\theta(o_t \mid q, o_{<t})}{\pi_{\theta_{\text{old}}}(o_t \mid q, o_{<t})} = \prod_{t=1}^T r_{i,t}(\theta)$$
在长文本生成（数百至数千 Token）中，如果使用累乘 $\prod_{t=1}^T r_{i,t}(\theta)$：只要策略发生微小漂移，连乘项就会发生严重的数值爆炸或消失（Variance Explosion），导致方差极大，训练崩溃。

---

### 3. GRPO 策略优化目标函数（Clipped Objective）

GRPO 最终优化的最大化目标函数 $\mathcal{J}_{\text{GRPO}}(\theta)$ 包含了带 PPO 裁剪机制的策略提升项，以及针对参考模型 $\pi_{\text{ref}}$ 的 KL 散度约束项：

$$\mathcal{J}_{\text{GRPO}}(\theta) = \mathbb{E}_{\substack{q \sim P(Q) \\ \{o_i\}_{i=1}^G \sim \pi_{\theta_{\text{old}}}}} \left[ \frac{1}{G} \sum_{i=1}^G \frac{1}{\vert{}o_i\vert{}} \sum_{t=1}^{\vert{}o_i\vert{}} \left( \min \left( r_{i,t}(\theta) A_i, \, \text{clip}\left(r_{i,t}(\theta), 1-\epsilon, 1+\epsilon\right) A_i \right) - \beta D_{\text{KL}}(\pi_\theta \parallel \pi_{\text{ref}}) \right) \right]$$

其中：

* **$\epsilon$**：PPO 裁剪范围超参数（通常设为 $0.1$ 或 $0.2$）。
* **$\beta$**：KL 惩罚系数（对应你代码设置中的 $\beta = 0.001$）。
* **$\vert{}o_i\vert{}$**：第 $i$ 个回答的序列长度（对长序列做平均归一化）。

---

### 4. KL 散度无偏/低方差估计器（KL Estimator）

为了在采样生成的序列上高效计算逐 Token 的 KL 散度，DeepSeek-R1 实操中常用 **Schulman 低方差无偏估计器**：

$$D_{\text{KL}}(\pi_\theta \parallel \pi_{\text{ref}}) = \frac{\pi_{\text{ref}}(o_{i,t} \mid q, o_{i,<t})}{\pi_\theta(o_{i,t} \mid q, o_{i,<t})} - \ln \frac{\pi_{\text{ref}}(o_{i,t} \mid q, o_{i,<t})}{\pi_\theta(o_{i,t} \mid q, o_{i,<t})} - 1$$

若令 $k = \frac{\pi_{\text{ref}}(o_{i,t} \mid q, o_{i,<t})}{\pi_\theta(o_{i,t} \mid q, o_{i,<t})}$，上式可简写为：


$$D_{\text{KL}}(\pi_\theta \parallel \pi_{\text{ref}}) = k - \ln k - 1$$

> 该估计器保证满足 $D_{\text{KL}} \ge 0$，且极小化单样本估计时的方差。

---

### 5. 梯度更新公式展开

当概率比率 $r_{i,t}(\theta)$ 处于未裁剪区间 $[1-\epsilon, 1+\epsilon]$ 内时，忽略 KL 项，单个 Token 的参数梯度简化为：

$$\nabla_\theta \mathcal{J}_{i,t}(\theta) = \nabla_\theta \ln \pi_\theta(o_{i,t} \mid q, o_{i,<t}) \cdot A_i$$

* **当 $A_i > 0$（表现优于组内平均水平）：** 增大生成该 Token 的对数概率。
* **当 $A_i < 0$（表现劣于组内平均水平）：** 降低生成该 Token 的对数概率。

本实验从已预训练的 Qwen3 Base 开始，不先进行本实验的 SFT，因此只是在小规模环境中学习 R1-Zero 式的可验证奖励训练流程，并非完整复现 DeepSeek-R1-Zero。


## 1. 先检查路径与环境

易贝云数据盘挂载到 /root/blockdata。模型/数据缓存、临时文件和输出统一放在该盘的 grpo_project 下；环境继续使用系统盘上的 /root/grpo-venv。上传 notebook 时同时上传 project_paths.py。

第一个代码单元必须在新内核中运行。数据盘未挂载时会停止，避免把30GB系统盘写满。


In [ ]:
# 第一个代码单元：先设置缓存路径，再导入 Hugging Face/TRL。
import sys
from pathlib import Path
if not Path("project_paths.py").exists() and Path("grpo/project_paths.py").exists():
    sys.path.insert(0, str(Path("grpo").resolve()))
from project_paths import setup_paths, require_space
paths = setup_paths()

import transformers, trl, torch, datasets

print(f"Transformers: {transformers.__version__}  (需 4.57.6)")
print(f"TRL:          {trl.__version__}  (本实验锁定 1.14.1)")
print(f"Datasets:     {datasets.__version__}")
print(f"PyTorch:      {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}, {p.total_memory / 1e9:.0f} GB")
assert trl.__version__ == "1.14.1", "请先安装 grpo/requirements.txt 中锁定的 TRL 版本"


## 2. 配置：按显卡选档位

先选下面的 `PROFILE`；显存占用受实际卡型、驱动、PyTorch 与 vLLM 版本影响，档位并非运行保证。想微调某个参数，直接改 `PROFILES` 或 `Config` 默认值。

| 档位 | 模型 | 训练方式 | G | 适用 |
|---|---|---|---|---|
| **`a800-80g`** | 1.7B-Base | 全参 | 8 | **A800-80GB（A100 平替，顶配）** |
| `a100-80g` | 1.7B-Base | 全参 | 8 | A100-80G |
| `a100-40g` | 1.7B-Base | 全参 | 4 | A100-40G / vGPU-48GB |
| `4090` | 0.6B-Base | QLoRA | 4 | RTX 4090 / 24G |
| `smoke` | 0.6B-Base | 全参 | 4 | 几分钟冒烟跑通 |

In [ ]:
from dataclasses import dataclass, asdict
import json
from pathlib import Path

# >>>>>>>>>>>>>>>>  只改这一行  <<<<<<<<<<<<<<<<
PREFLIGHT = False       # True: 用正式模型/G/vLLM只训练5步；与正式输出分开
PROFILE = "a800-80g"     # a800-80g / a100-80g / a100-40g / 4090 / smoke

BASE_MODEL_ID = "Qwen/Qwen3-1.7B-Base"#基座模型
DISTILL_MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"#蒸馏模型
INSTRUCT_MODEL_ID = "Qwen/Qwen3-1.7B"#指令模型
GSM8K_ID = "openai/gsm8k"#GSM8K数据集

PROMPT_TEMPLATE = (
    "Solve the following math problem step by step. "
    "Put your final answer in \\boxed{{}}.\n\n"
    "Problem: {question}\n\n"
    "Solution:"
)

OUTPUT_DIR = paths["outputs"]  # 云端: /root/blockdata/grpo_project/outputs


@dataclass
class Config:
    model_id: str = BASE_MODEL_ID
    run_name: str = "grpo-qwen3-1.7b-math"
    training_mode: str = "full"          # full / lora / qlora
    num_generations: int = 8             # 每个问题生成的答案数量
    max_prompt_length: int = 512
    max_completion_length: int = 1024
    beta: float = 0.001                  # KL 系数；新版仍用 beta，0 表示不计算参考模型 KL
    learning_rate: float = 5e-6
    per_device_train_batch_size: int = 1 # 每个 GPU 的训练 batch size
    gradient_accumulation_steps: int = 8 # 单卡有效 batch=1×8，可被默认 G=8 整除
    max_steps: int = 500
    num_train_epochs: float = 1.0
    temperature: float = 1.0
    top_p: float = 0.95
    bf16: bool = True
    gradient_checkpointing: bool = True
    attn_implementation: str = "flash_attention_2"# 指定使用哪种注意力实现方式，flash_attention_2 是最快的
    mask_truncated_completions: bool = True # 是否屏蔽截断的完成
    loss_type: str = "bnpo"
    scale_rewards: bool = True
    # 是否使用 vllm
    use_vllm: bool = False #VLLM可以显著提高推理速度，尤其是在生成多个答案时。
    vllm_mode: str = "colocate"  #colocate将vllm和训练脚本放在同一台机器上运行，减少网络传输延迟。另一种模式是"server"，由独立vLLM服务生成。
    vllm_gpu_memory_utilization: float = 0.3
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.0
    logging_steps: int = 1
    save_steps: int = 100
    save_total_limit: int = 1
    save_only_model: bool = True        # 30GB盘：只保存模型，不保存优化器；不能完整续训
    seed: int = 42
    max_train_samples: int | None = None
    num_test_samples: int = 100
    eval_max_new_tokens: int = 2048


PROFILES = {
    "a800-80g": {
        "model_id": BASE_MODEL_ID, "run_name": "grpo-qwen3-1.7b-math",
        "training_mode": "full", "num_generations": 8,
        "max_completion_length": 1024, "max_steps": 500,
        "use_vllm": True, "vllm_mode": "colocate", "vllm_gpu_memory_utilization": 0.3,
    },
    "a100-80g": {
        "model_id": BASE_MODEL_ID, "run_name": "grpo-qwen3-1.7b-math",
        "training_mode": "full", "num_generations": 8,
        "max_completion_length": 1024, "max_steps": 500,
        "use_vllm": True, "vllm_mode": "colocate", "vllm_gpu_memory_utilization": 0.3,
    },
    "a100-40g": {
        "model_id": BASE_MODEL_ID, "run_name": "grpo-qwen3-1.7b-math-g4",
        "training_mode": "full", "num_generations": 4,
        "max_completion_length": 1024, "max_steps": 500,
        "gradient_accumulation_steps": 8,
        "use_vllm": True, "vllm_gpu_memory_utilization": 0.25,
    },
    "4090": {
        "model_id": "Qwen/Qwen3-0.6B-Base", "run_name": "grpo-qwen3-0.6b-math-qlora",
        "training_mode": "qlora", "num_generations": 4,
        "max_completion_length": 768, "max_steps": 500,
        "gradient_accumulation_steps": 8, "use_vllm": False,
    },
    "smoke": {
        "model_id": "Qwen/Qwen3-0.6B-Base", "run_name": "smoke",
        "training_mode": "full", "num_generations": 4,
        "max_completion_length": 256, "max_steps": 5,
        "max_train_samples": 32, "save_steps": 5,
        "use_vllm": False, "gradient_checkpointing": False,
    },
}


def get_config(profile: str) -> Config:
    """输入档位名，返回独立 Config；未知档位抛 KeyError。"""
    c = Config()
    for k, v in PROFILES[profile].items():
        setattr(c, k, v)
    return c


cfg = get_config(PROFILE)
if PREFLIGHT:
    cfg.run_name += "-preflight"
    cfg.max_steps = 5
    cfg.max_train_samples = 64
    cfg.save_steps = 5
run_dir = OUTPUT_DIR / cfg.run_name
final_model_dir = run_dir / "final"
run_dir.mkdir(parents=True, exist_ok=True)
print(f"档位 {PROFILE}：model={cfg.model_id}, G={cfg.num_generations}, "
      f"mode={cfg.training_mode}, steps={cfg.max_steps}, vllm={cfg.use_vllm}")

## 3. 加载 GSM8K 数据集

GSM8K 答案格式为：**推理过程 + `#### <最终数值>`**。

In [ ]:
from datasets import load_dataset

gsm8k = load_dataset(GSM8K_ID, "main")
print(f"训练集: {len(gsm8k['train'])} 条")
print(f"测试集: {len(gsm8k['test'])} 条")

sample = gsm8k["train"][0]
print("\n问题:\n", sample["question"])
print("\n答案:\n", sample["answer"])

## 4. Step 1：实现奖励函数

### 4.1 从 GSM8K 标准答案提取最终数值

In [ ]:
import re

HASH_RE = re.compile(r"####\s*(-?[\d,]*\.?\d+)")
NUMBER_RE = re.compile(r"-?\d[\d,]*\.?\d*")


def normalize_number(text) -> str:
    """输入答案文本，返回去掉常见货币符号与千分位逗号的字符串。"""
    # 去千分位逗号、货币/百分号、空白
    text = str(text).strip().replace(",", "")
    return text.strip("$￥£€=:{}()（）").replace("%", "").strip()


def extract_gsm8k_answer(answer_text: str) -> str:
    """输入 GSM8K answer，优先解析 #### 后数值，返回标准答案字符串。"""
    m = HASH_RE.search(str(answer_text))
    if m:
        return normalize_number(m.group(1))
    nums = NUMBER_RE.findall(str(answer_text))
    return normalize_number(nums[-1]) if nums else ""


for i in range(5):
    gt = extract_gsm8k_answer(gsm8k["train"][i]["answer"])
    print(f"问题 {i+1} 标准答案: {gt}")

### 4.2 从模型回复提取答案

优先级：`\boxed{}` → "答案是 X / answer is X" → `#### X` → 最后一个数字。
额外处理：千分位、负数、小数、`\frac{}` 分数、`$`、`%`。
末尾数字回退可能把推理过程中的中间值当作最终答案。正式评测建议统计解析来源，并人工抽查误判。


In [ ]:
BOXED_RE = re.compile(r"\\boxed\s*\{((?:[^{}]|\{[^{}]*\})*)\}")
ANSWER_IS_RE = re.compile(
    r"(?:答案|结果|最终答案|answer(?:\s+is)?|result(?:\s+is)?)"
    r"[\s:：是为=]*(-?\d[\d,]*\.?\d*)",
    re.IGNORECASE,
)


def eval_latex_number(raw: str) -> str:
    """输入简单 LaTeX 数值/分数，返回可比较的数值字符串。"""
    # \frac{1}{2} -> 0.5
    frac = re.match(
        r"\\[dt]?frac\s*\{\s*(-?\d+(?:\.\d+)?)\s*\}\s*\{\s*(-?\d+(?:\.\d+)?)\s*\}$",
        raw.strip(),
    )
    if frac:
        num, den = float(frac.group(1)), float(frac.group(2))
        if den != 0:
            v = num / den
            return str(int(v)) if v.is_integer() else str(v)
    return normalize_number(raw)


def extract_answer_from_response(response: str) -> str:
    """输入生成文本，按 boxed/答案短语/####/末尾数字提取答案；末尾数字仅作宽松回退。"""
    text = response if isinstance(response, str) else str(response)
    b = BOXED_RE.search(text)
    if b:
        v = eval_latex_number(b.group(1))
        if NUMBER_RE.fullmatch(v.replace(" ", "")):
            return normalize_number(v)
        inner = NUMBER_RE.search(v)
        return normalize_number(inner.group(0)) if inner else v
    a = ANSWER_IS_RE.search(text)
    if a:
        return normalize_number(a.group(1))
    h = HASH_RE.search(text)
    if h:
        return normalize_number(h.group(1))
    nums = NUMBER_RE.findall(text)
    return normalize_number(nums[-1]) if nums else ""


# 快速验证
for txt in [r"\boxed{42}", "答案是 15", r"\boxed{1,234}", r"\boxed{\frac{1}{2}}", r"\boxed{-3}"]:
    print(f"{txt:<20} -> {extract_answer_from_response(txt)}")

### 4.3 正确性奖励与可选格式奖励

默认只训练正确性奖励：答案对为 1，错为 0。格式奖励函数作为教学示例保留，但默认未交给 Trainer。Base 模型不保证输出思考标签，是否使用格式分要先检查真实生成样本。

课程代码中的 for completion in zip(completions) 会得到单元素元组，这里已改为直接遍历生成文本。


In [ ]:
THINK_OPEN = "<think>"
THINK_CLOSE = "</think>"


def completion_to_text(completion) -> str:
    """输入 TRL completion（字符串或消息列表），返回纯文本。"""
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list):
        return "".join(
            str(t.get("content", "")) if isinstance(t, dict) else str(t)
            for t in completion
        )
    return str(completion)


def is_close(pred, gt, tol=1e-4) -> bool:
    """输入预测和标准答案，返回容差内数值是否相等。"""
    try:
        return abs(float(pred) - float(gt)) <= tol
    except (ValueError, TypeError):
        return pred.strip() == gt.strip()


def correctness_reward(completions, answer, **kwargs):
    """输入一批生成和对应 GSM8K 原始答案，返回逐条 0/1 正确性奖励。"""
    rewards = []
    for completion, ans_text in zip(completions, answer):
        gt = extract_gsm8k_answer(ans_text)
        pred = extract_answer_from_response(completion_to_text(completion))
        rewards.append(1.0 if gt != "" and is_close(pred, gt) else 0.0)
    return rewards


def format_reward(completions, **kwargs):
    """输入一批生成，返回逐条格式分；默认训练未启用此函数。"""
    rewards = []
    for completion in completions:
        text = completion_to_text(completion)
        r = 0.0
        if THINK_OPEN in text and THINK_CLOSE in text:
            r += 0.1
        if BOXED_RE.search(text):
            r += 0.1
        rewards.append(r)
    return rewards

### 4.4 验证奖励函数（单元测试）

全部 PASS 才能进入下一步——奖励函数是 GRPO 最容易出 bug 的地方。

In [ ]:
test_cases = [
    (r"Let me think...\nThe answer is \boxed{42}", "reasoning\n#### 42", 1.0),
    ("I think the answer is 100", "reasoning\n#### 50", 0.0),
    ("经过计算，答案是 15", "reasoning\n#### 15", 1.0),
    ("I'm not sure, maybe 7?", "reasoning\n#### 7", 1.0),
    ("This is a hard problem with no number", "reasoning\n#### 25", 0.0),
    (r"The result is \boxed{1,234}", "reasoning\n#### 1234", 1.0),
    (r"\boxed{\frac{1}{2}}", "reasoning\n#### 0.5", 1.0),
    (r"\boxed{-3}", "reasoning\n#### -3", 1.0),
]

all_pass = True
for i, (resp, ans, expected) in enumerate(test_cases):
    rw = correctness_reward([resp], [ans])[0]
    ok = rw == expected
    all_pass &= ok
    print(f"[{'PASS' if ok else 'FAIL'}] Case {i+1}: reward={rw}")

fmt_cases = [
    ("<think>x</think>\n\\boxed{1}", 0.2),
    (r"\boxed{1}", 0.1),
    ("plain text", 0.0),
]
for i, (resp, expected) in enumerate(fmt_cases):
    rw = format_reward([resp])[0]
    ok = rw == expected
    all_pass &= ok
    print(f"[{'PASS' if ok else 'FAIL'}] Format Case {i+1}: reward={rw}")

print("\n全部通过 ✅" if all_pass else "\n有失败用例，先修奖励函数！")

## 5. Step 2：加载基座模型

按 `training_mode` 选择 full / lora / qlora；请求 flash_attention_2 但未安装时自动回退 sdpa。

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def load_tokenizer(model_id):
    """输入模型 ID/目录，返回适合 decoder-only 批生成的 tokenizer。"""
    tok = AutoTokenizer.from_pretrained(model_id, padding_side="left")
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok


def resolve_attn(requested):
    """输入注意力实现名；FlashAttention 不可用时返回 sdpa。"""
    if requested == "flash_attention_2":
        try:
            import flash_attn  # noqa
            return "flash_attention_2"
        except Exception:
            print("[模型] 未装 flash-attn，回退 sdpa")
            return "sdpa"
    return requested


def make_lora_config(c):
    """输入训练配置，返回对应的 PEFT LoRA 配置。"""
    from peft import LoraConfig
    return LoraConfig(
        task_type="CAUSAL_LM", r=c.lora_r, lora_alpha=c.lora_alpha,
        lora_dropout=c.lora_dropout, target_modules="all-linear", bias="none",
    )


def load_model(cfg):
    """输入训练配置，返回 (模型, tokenizer)；QLoRA 路径加载 4 bit 基座。"""
    tok = load_tokenizer(cfg.model_id)
    dtype = torch.bfloat16 if cfg.bf16 else torch.float16
    attn = resolve_attn(cfg.attn_implementation)

    if cfg.training_mode == "qlora":
        from transformers import BitsAndBytesConfig
        from peft import get_peft_model, prepare_model_for_kbit_training
        qc = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype,
        )
        m = AutoModelForCausalLM.from_pretrained(
            cfg.model_id, quantization_config=qc, device_map={"": 0}, attn_implementation=attn)
        m = prepare_model_for_kbit_training(m, use_gradient_checkpointing=cfg.gradient_checkpointing)
        m = get_peft_model(m, make_lora_config(cfg))
    elif cfg.training_mode == "lora":
        from peft import get_peft_model
        m = AutoModelForCausalLM.from_pretrained(
            cfg.model_id, device_map={"": 0}, dtype=dtype, attn_implementation=attn)
        m = get_peft_model(m, make_lora_config(cfg))
    else:
        m = AutoModelForCausalLM.from_pretrained(
            cfg.model_id, device_map={"": 0}, dtype=dtype, attn_implementation=attn)
    m.config.pad_token_id = tok.pad_token_id
    print(f"[模型] {cfg.model_id} 加载完成（{cfg.training_mode}）")
    return m, tok




In [ ]:
# 第5节：加载训练模型，只评估已有权重时跳过
model, tokenizer = load_model(cfg)


### 5.1 统一生成工具（分析 / 对比复用）

不同模型用它最匹配的输入方式：Base/GRPO 用纯文本 prompt；Distill/Instruct 用 chat template（Instruct 开启思考）。

In [ ]:
@torch.no_grad()
def generate_text(m, tok, prompt_text, max_new_tokens=1024, temperature=0.0, top_p=0.95):
    """输入模型、tokenizer 和 prompt，返回新增文本；temperature=0 为贪心解码。"""
    inputs = tok(prompt_text, return_tensors="pt").to(m.device)
    do_sample = temperature > 0
    gk = {"max_new_tokens": max_new_tokens, "do_sample": do_sample, "pad_token_id": tok.pad_token_id}
    if do_sample:
        gk.update(temperature=temperature, top_p=top_p)
    out = m.generate(**inputs, **gk)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


def make_base_prompt(question):
    """输入 GSM8K 问题，返回 Base 模型训练/评估用纯文本提示。"""
    return PROMPT_TEMPLATE.format(question=question)


def make_chat_prompt(tok, question, enable_thinking=True):
    """输入 tokenizer 和问题，返回 Instruct/Distill 的 chat 模板提示。"""
    messages = [{"role": "user",
                 "content": f"Solve the following math problem. Put your final answer in \\boxed{{}}.\n\n{question}"}]
    kw = dict(tokenize=False, add_generation_prompt=True)
    if enable_thinking is not None:
        kw["enable_thinking"] = enable_thinking
    return tok.apply_chat_template(messages, **kw)

### 5.2 准备训练数据

In [ ]:
def format_example(ex):
    """输入 GSM8K 样本，返回包含 prompt 和原始 answer 的 GRPO 样本。"""
    return {"prompt": make_base_prompt(ex["question"]), "answer": ex["answer"]}


train_split = gsm8k["train"]
if cfg.max_train_samples is not None:
    train_split = train_split.select(range(min(cfg.max_train_samples, len(train_split))))
train_dataset = train_split.map(format_example, remove_columns=train_split.column_names)
# 新版 TRL 不再提供 max_prompt_length；显式过滤过长提示，避免静默截断题目。
train_dataset = train_dataset.filter(
    lambda ex: len(tokenizer(ex["prompt"], add_special_tokens=False)["input_ids"]) <= cfg.max_prompt_length
)
assert len(train_dataset) > 0, "过滤后训练集为空，请检查 max_prompt_length"
print(f"训练数据量: {len(train_dataset)}")
print("\n示例提示:\n", train_dataset[0]["prompt"])

## 6. 配置 GRPO 并启动训练

本项目采用30GB数据盘的节省空间设置：每100步保存一次模型检查点，只保留最近一份；训练结束另存最终权重。save_only_model=True 不保存优化器和调度器，不能用这些检查点完整恢复训练。若要完整续训，先扩容数据盘，再关闭此设置。

正式训练前先使用 PROFILE="smoke"；之后保持 PROFILE="a800-80g"，设 PREFLIGHT=True 验证正式模型的5步训练；通过后重启内核，改回 PREFLIGHT=False。


In [ ]:
from trl import GRPOConfig, GRPOTrainer

grpo_config = GRPOConfig(
    output_dir=str(run_dir),
    num_generations=cfg.num_generations,
    max_completion_length=cfg.max_completion_length,
    temperature=cfg.temperature,
    top_p=cfg.top_p,
    beta=cfg.beta,
    learning_rate=cfg.learning_rate,
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    num_train_epochs=cfg.num_train_epochs,
    max_steps=cfg.max_steps,
    mask_truncated_completions=cfg.mask_truncated_completions,
    loss_type=cfg.loss_type,
    scale_rewards=cfg.scale_rewards,
    bf16=cfg.bf16,
    gradient_checkpointing=cfg.gradient_checkpointing,
    use_vllm=cfg.use_vllm,
    vllm_mode=cfg.vllm_mode,
    vllm_gpu_memory_utilization=cfg.vllm_gpu_memory_utilization,
    logging_steps=cfg.logging_steps,
    save_steps=cfg.save_steps,
    save_total_limit=cfg.save_total_limit,
    save_only_model=cfg.save_only_model,
    seed=cfg.seed,
    report_to="none",
    remove_unused_columns=False,   # 保留 answer 列
    log_completions=True,         # 打印生成样本，观察推理演变
)
print("GRPOConfig 就绪")

In [ ]:
# 给检查点写入、旧检查点轮换以及最终权重预留空间。
require_space(run_dir, 8)
if (final_model_dir / "config.json").exists() or (final_model_dir / "adapter_config.json").exists():
    raise RuntimeError("本实验已有最终权重；先更改 cfg.run_name，避免覆盖")

trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    train_dataset=train_dataset,
    reward_funcs=[correctness_reward],   # 想加格式分：[correctness_reward, format_reward]
    processing_class=tokenizer,
)
print("=" * 60)
print(f"开始训练：{cfg.model_id} | G={cfg.num_generations} | steps={cfg.max_steps}")
print(f"beta={cfg.beta} | lr={cfg.learning_rate} | vLLM={cfg.use_vllm}")
print("=" * 60)

result = trainer.train()
print(f"\n训练完成，总步数 {result.global_step}")
require_space(run_dir, 4)
trainer.save_model(str(final_model_dir))
tokenizer.save_pretrained(str(final_model_dir))
trainer.save_state()
trainer.log_metrics("train", result.metrics)
trainer.save_metrics("train", result.metrics)
(run_dir / "experiment_config.json").write_text(
    json.dumps(asdict(cfg), ensure_ascii=False, indent=2), encoding="utf-8")
(run_dir / "log_history.json").write_text(
    json.dumps(trainer.state.log_history, ensure_ascii=False, indent=2), encoding="utf-8")
print("最终模型已保存到：", final_model_dir)

## 7. 查看训练日志

从 Trainer 的实际日志提取奖励、组内差异、输出长度及 KL（仅启用 KL 时可能存在）。这些是诊断信号，不等于独立测试集准确率；新版 TRL 的日志键名若变化，缺失曲线会留空。


In [ ]:
import matplotlib.pyplot as plt  # Notebook 保持默认后端，保存图片时也能内联显示


def series(log_history, key):
    """输入 Trainer 日志和指标名，返回同指标自己的 (steps, values)。"""
    xs, ys = [], []
    for e in log_history:
        if key in e and e[key] is not None:
            xs.append(e.get("step", len(xs)))
            ys.append(e[key])
    return xs, ys


log_history = trainer.state.log_history
r_steps, r_mean = series(log_history, "reward")
std_steps, r_std = series(log_history, "reward_std")
length_steps, lengths = series(log_history, "completions/mean_length")
kl_steps, kls = series(log_history, "kl")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0, 0].plot(r_steps, r_mean, "b-"); axes[0, 0].set_title("Reward Mean (accuracy)")
axes[0, 0].set_ylim(-0.02, 1.05); axes[0, 0].grid(alpha=0.3)
axes[0, 1].plot(std_steps, r_std, "r-"); axes[0, 1].set_title("Reward Std (diversity)")
axes[0, 1].grid(alpha=0.3)
axes[1, 0].plot(length_steps, lengths, "g-"); axes[1, 0].set_title("Response Length (depth)")
axes[1, 0].grid(alpha=0.3)
axes[1, 1].plot(kl_steps, kls, "m-"); axes[1, 1].set_title("KL Divergence")
axes[1, 1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(run_dir / "grpo_training_curves.png", dpi=150)
plt.show()
print("曲线已保存：", run_dir / "grpo_training_curves.png")

### 指标怎么读

训练奖励反映当前采样回答的判分，不等于测试集准确率；应同时看独立测试题。组内奖励标准差长期为 0 时，常见原因是同题样本全对或全错。输出长度突然增长时，检查截断和重复生成。KL 仅在 beta>0 且日志提供该指标时绘制；它的量级没有通用阈值，应与同配置实验及实际回答一起分析。

每题生成 G 个回复是主要成本之一；vLLM 可能加速生成，也会额外占用显存，需在目标机器上实测。


## 8. 观察生成行为

用固定样例查看模型是否逐步计算、自检、改写答案。下方关键词命中只是粗略文本线索，不能证明推理能力“涌现”；判断效果仍要看独立题集的逐题正确率。


In [ ]:
TEST_QUESTIONS = [
    "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. "
    "How many clips did Natalia sell altogether in April and May?",
    "Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?",
    "Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. "
    "Her parents decided to give her $15, and her grandparents twice as much as her parents. "
    "How much more money does Betty need to buy the wallet?",
    "Julie is reading a 120-page book. Yesterday she read 12 pages and today she read twice as many. "
    "If she wants to read half of the remaining pages tomorrow, how many pages should she read?",
    "James writes a 3-page letter to 2 different friends twice a week. How many pages does he write a year?",
]

EMERGENCE_CHECKLIST = {
    "step_by_step": ["step", "first", "then", "next", "finally", "首先", "然后", "接下来"],
    "self_check": ["verify", "check", "double-check", "确认", "验证", "检查"],
    "backtrack": ["wait", "actually", "reconsider", "mistake", "不对", "重新", "等等"],
    "structure": ["therefore", "thus", "because", "hence", "因此", "所以", "于是"],
    "math_symbol": ["\\boxed", "=", "+", "-", "*", "/"],
}


def analyze(m, tok, questions, temperature=0.0):
    """输入模型与问题列表，返回生成文本、提取答案及关键词命中记录。"""
    records = []
    for i, q in enumerate(questions):
        resp = generate_text(m, tok, make_base_prompt(q),
                             max_new_tokens=cfg.eval_max_new_tokens, temperature=temperature)
        behaviors = {n: [kw for kw in kws if kw.lower() in resp.lower()]
                     for n, kws in EMERGENCE_CHECKLIST.items()}
        records.append(dict(index=i + 1, question=q, response=resp,
                            num_words=len(resp.split()),
                            predicted=extract_answer_from_response(resp), behaviors=behaviors))
    return records


def load_saved_model(source, base_id, dtype):
    """输入保存目录和原基座 ID，返回可推理模型；自动识别 PEFT adapter。"""
    from pathlib import Path
    adapter_file = Path(source) / "adapter_config.json"
    if adapter_file.exists():
        from peft import PeftModel
        base = AutoModelForCausalLM.from_pretrained(
            base_id, dtype=dtype, device_map={"": 0})
        return PeftModel.from_pretrained(base, source)
    return AutoModelForCausalLM.from_pretrained(
        source, dtype=dtype, device_map={"": 0})




In [ ]:
# 第8节：可选样例分析，重启后只评估时跳过
# 训练及绘图已经完成；先释放原训练模型，避免重复加载导致显存不足。
import gc
for name in ("trainer", "model"):
    globals().pop(name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

grpo_model = load_saved_model(
    str(final_model_dir), cfg.model_id, torch.bfloat16 if cfg.bf16 else torch.float16)
grpo_model.eval()
records = analyze(grpo_model, tokenizer, TEST_QUESTIONS)

n = len(records)
for name in EMERGENCE_CHECKLIST:
    cnt = sum(1 for r in records if r["behaviors"][name])
    print(f"{name:<14}{cnt}/{n}")

for r in records[:2]:
    print("=" * 60)
    print("问题", r["index"], ":", r["question"][:90])
    print(r["response"][:700])
    print("提取答案:", r["predicted"])

## 9. 模型评估（100题，四个模型）

重启后只运行：第1节环境 → 第2节配置（保持 a800-80g，读取原1.7B实验）→ 第3节数据 → 第4节解析/奖励函数 → 第5节函数定义（不要运行加载训练模型）→ 第5.1节生成工具 → 第8节函数定义（不要运行样例分析）→ 本节评估。

4090只改变硬件，不要改训练PROFILE，否则会指向0.6B实验。四模型依次BF16加载，每完成一个立即保存逐题问题、回答和判分；100题写到新目录，不覆盖20题。解码仍为贪心，生成预算1024，测试集前100题。

预算2048；保留触顶记录并从未触顶准确率中排除，同时报告全题准确率、触顶数和真截断数。不同模型排除题目不同，未触顶准确率不能直接作公平排名。明确答案取最后一个，末尾数字仅审计，不参与正式判分。旧训练奖励与解析实现不修改。


In [ ]:
import gc, csv
from tqdm.auto import tqdm


def parse_eval_answer(text):
    """输入完整回复，返回明确答案及解析来源；末尾数字仅供审计，不用于正式判分。"""
    # 闭合思考标签后优先只看最终回答；只有未闭合思考时不猜中间结果。
    if "</think>" in text:
        final = text.rsplit("</think>", 1)[1]
    elif "<think>" in text:
        return None, "unfinished_think"
    else:
        final = text
    boxed = list(BOXED_RE.finditer(final))
    if boxed:
        value = eval_latex_number(boxed[-1].group(1))
        if NUMBER_RE.fullmatch(value):
            return value, "last_boxed"
        return None, "invalid_boxed"
    hashes = list(HASH_RE.finditer(final))
    if hashes:
        return normalize_number(hashes[-1].group(1)), "hash"
    phrases = list(ANSWER_IS_RE.finditer(final))
    if phrases:
        return normalize_number(phrases[-1].group(1)), "answer_phrase"
    if NUMBER_RE.fullmatch(final.strip()):
        return normalize_number(final.strip()), "number_only"
    return None, "no_explicit_answer"


@torch.inference_mode()
def generate_eval_record(m, tok, prompt, max_tokens):
    """输入模型、tokenizer、提示和预算，返回文本、生成token数、触顶及EOS标记。"""
    inputs = tok(prompt, return_tensors="pt").to(m.device)
    output = m.generate(**inputs, do_sample=False, max_new_tokens=max_tokens,
                        pad_token_id=tok.pad_token_id, use_cache=True)
    ids = output[0, inputs["input_ids"].shape[1]:]
    eos = m.generation_config.eos_token_id
    eos_ids = eos if isinstance(eos, list) else [eos]
    ended_with_eos = bool(len(ids) and int(ids[-1]) in eos_ids)
    return dict(response=tok.decode(ids, skip_special_tokens=True),
                generated_tokens=len(ids), hit_limit=len(ids) >= max_tokens,
                ended_with_eos=ended_with_eos,
                truncated=len(ids) >= max_tokens and not ended_with_eos)


def evaluate_one(name, source, kind, questions, answers):
    """输入模型来源及题目，逐题生成并返回准确率和明细；支持 PEFT adapter。"""
    print(f"\n>>> 评测 {name}（{source}）", flush=True)
    tok = load_tokenizer(cfg.model_id if name == "GRPO" else source)
    m = load_saved_model(source, cfg.model_id, torch.bfloat16 if cfg.bf16 else torch.float16)
    m.eval()
    details, correct = [], 0
    for i, (q, a) in enumerate(tqdm(zip(questions, answers), total=len(questions), desc=name, unit="题")):
        if kind == "base":
            prompt = make_base_prompt(q)
        elif kind == "chat":
            prompt = make_chat_prompt(tok, q, enable_thinking=None)
        else:
            prompt = make_chat_prompt(tok, q, enable_thinking=True)
        generation = generate_eval_record(m, tok, prompt, cfg.eval_max_new_tokens)
        resp = generation["response"]
        pred, parse_source = parse_eval_answer(resp)
        gt = extract_gsm8k_answer(a)
        ok = pred is not None and is_close(pred, gt)
        correct += int(ok)
        details.append(dict(index=i, question=q, ground_truth=gt, predicted=pred,
                            correct=bool(ok), parse_source=parse_source,
                            fallback_prediction=extract_answer_from_response(resp), **generation))
    del m, tok
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    eligible = [r for r in details if not r["hit_limit"]]
    eligible_correct = sum(r["correct"] for r in eligible)
    return dict(name=name, correct=correct, total=len(questions),
                accuracy=correct / len(questions),
                eligible_total=len(eligible), eligible_correct=eligible_correct,
                accuracy_non_limit=eligible_correct / len(eligible) if eligible else None,
                hit_limit_count=sum(r["hit_limit"] for r in details),
                truncated_count=sum(r["truncated"] for r in details),
                parse_ok_count=sum(r["predicted"] is not None for r in details), details=details)


test_raw = gsm8k["test"].select(range(min(cfg.num_test_samples, len(gsm8k["test"]))))
questions = [e["question"] for e in test_raw]
answers = [e["answer"] for e in test_raw]

models = [
    ("Base", cfg.model_id, "base"),
    ("GRPO", str(final_model_dir), "base"),

]
# 30GB盘默认只评估当前 Base 和 GRPO。参考模型需额外缓存约6GB权重。
RUN_REFERENCE_MODELS = True
if RUN_REFERENCE_MODELS:
    require_space(run_dir, 0.5)  # 参考模型已经缓存；此处预留结果写入空间
    models += [("Distill", DISTILL_MODEL_ID, "chat"),
               ("Instruct-Think", INSTRUCT_MODEL_ID, "chat-think")]
# 上一节的分析模型占显存；评估前释放，依次加载四个模型。
globals().pop("grpo_model", None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
eval_dir = run_dir / f"eval_test_{len(questions)}_greedy_{cfg.eval_max_new_tokens}_strict_v2"
eval_dir.mkdir(parents=True, exist_ok=True)
results = []
for name, source, kind in models:
    result = evaluate_one(name, source, kind, questions, answers)
    results.append(result)
    # 每完成一个模型立即保存，网页断线也不丢结果；与20题结果分目录。
    (eval_dir / f"{name}.json").write_text(
        json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
with (eval_dir / "summary.csv").open("w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "correct", "total", "accuracy", "eligible_correct", "eligible_total", "accuracy_non_limit", "hit_limit_count", "truncated_count", "parse_ok_count"])
    writer.writeheader()
    writer.writerows({key: r[key] for key in writer.fieldnames} for r in results)
(eval_dir / "protocol.json").write_text(json.dumps({
    "dataset": GSM8K_ID, "split": "test", "indices": list(range(len(questions))),
    "parser": "strict_v2_last_explicit_answer; no last-number fallback scoring",
    "exclusion": "hit_limit excluded only from accuracy_non_limit; retained in all-question accuracy",
    "do_sample": False, "max_new_tokens": cfg.eval_max_new_tokens,
    "dtype": "bfloat16" if cfg.bf16 else "float16", "base_prompt": PROMPT_TEMPLATE,
    "models": models, "note": "参考模型使用各自chat模板；非严格消融"
}, ensure_ascii=False, indent=2), encoding="utf-8")
print("评估结果已保存:", eval_dir)

print("\n" + "=" * 50)
print(f"GSM8K 准确率（{len(questions)} 题，贪心）")
print("=" * 50)
for r in results:
    filtered = "无可计分题" if r["accuracy_non_limit"] is None else f"{r['accuracy_non_limit']*100:.1f}%"
    print(f"{r['name']}: 未触顶 {r['eligible_correct']}/{r['eligible_total']} ({filtered}); "
          f"全题 {r['correct']}/{r['total']} ({r['accuracy']*100:.1f}%); "
          f"触顶 {r['hit_limit_count']}, 真截断 {r['truncated_count']}, 可解析 {r['parse_ok_count']}")

### 如何读四模型结果

以本次运行得到的正确题数/总题数为准，不预设分数范围。Base/GRPO 用纯文本提示，Distill/Instruct 用 chat 模板，因此四者的数字是探索性对照，不能把差值全部归因于 GRPO。当前评估 100 道题，抽样误差很大；正式比较应固定完整 GSM8K test、生成预算和多随机种子。


## 10. 可选：改变每题生成数 G

下方实验会额外训练三个模型，默认不运行。G=4/8/16 会改变采样成本和有效 batch，结果只能作为探索，不能称为严格消融。单卡有效 batch 必须可被 G 整除，函数会相应增大梯度累积数。


In [ ]:
import json


def run_for_g(g, max_steps):
    """输入 G 和步数，从基座独立训练一次，返回末次训练奖励。"""
    c = get_config(PROFILE)
    c.model_id = cfg.model_id
    c.num_generations = g
    c.max_steps = max_steps
    c.gradient_accumulation_steps = max(c.gradient_accumulation_steps, g)
    c.use_vllm = False
    rd = OUTPUT_DIR / f"ablation-g{g}"
    rd.mkdir(parents=True, exist_ok=True)
    m, tok = load_model(c)
    gc = GRPOConfig(
        output_dir=str(rd), num_generations=g,
        max_completion_length=c.max_completion_length,
        temperature=c.temperature, top_p=c.top_p, beta=c.beta,
        learning_rate=c.learning_rate, per_device_train_batch_size=c.per_device_train_batch_size,
        gradient_accumulation_steps=c.gradient_accumulation_steps, max_steps=max_steps,
        scale_rewards=c.scale_rewards,
        mask_truncated_completions=c.mask_truncated_completions, loss_type=c.loss_type,
        bf16=c.bf16, gradient_checkpointing=c.gradient_checkpointing,
        use_vllm=False, save_strategy="no", seed=c.seed, report_to="none",
        remove_unused_columns=False)
    tr = GRPOTrainer(model=m, args=gc, train_dataset=train_dataset,
                     reward_funcs=[correctness_reward], processing_class=tok)
    tr.train()
    rs = [e["reward"] for e in tr.state.log_history if "reward" in e]
    del tr, m, tok
    import gc as garbage_collector
    garbage_collector.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return dict(g=g, final_reward=rs[-1] if rs else None)


RUN_G_EXPERIMENTS = False  # 需要额外训练三次；手动开启。
if RUN_G_EXPERIMENTS:
    ablation_results = [run_for_g(g, 20) for g in [4, 8, 16]]
    for r in ablation_results:
        print(f"G={r['g']:<4}最终奖励={r['final_reward']}")
    with open(OUTPUT_DIR / "ablation_g_results.json", "w", encoding="utf-8") as f:
        json.dump(ablation_results, f, ensure_ascii=False, indent=2)

## 11. 完成后记录什么

1. 保存训练配置、依赖版本、随机种子、硬件和运行耗时。
2. 查看训练奖励、组内差异、长度与 KL 曲线，但不要把关键词命中当成推理能力证据。
3. 在固定的独立 GSM8K test 题集上记录正确题数、总题数和逐题输出；正式结论应扩大样本并重复种子。
4. 对比 Base、GRPO、Distill 与 Instruct 时标注提示模板和训练数据不同，避免把全部差异归因于 GRPO。
5. 记录全对/全错组比例、解析器回退次数和截断率，帮助判断奖励信号是否可靠。

本文件的 loss_type 明确设为 bnpo；TRL 1.14.1 的默认值为 dapo。这里只升级接口并保留原实验目标函数，不代表复现完整 DAPO。


In [ ]:
# 只重评最后一个模型：需先运行过第9节函数定义；中断而未重启内核可直接运行。
import gc, sys, time
from tqdm.auto import tqdm


def evaluate_instruct_batches(batch_size=4):
    """输入推理batch大小；续跑Instruct-Think，逐批保存，并读取前三模型JSON生成汇总。"""
    directory = run_dir / f"eval_test_100_greedy_{cfg.eval_max_new_tokens}_strict_v2"
    assert cfg.num_test_samples == 100 and cfg.eval_max_new_tokens == 2048
    for name in ("Base", "GRPO", "Distill"):
        assert (directory / f"{name}.json").is_file(), f"缺少已完成结果: {name}"
    # 释放中断留下的异常堆栈，避免堆栈继续引用旧模型占用显存。
    for name in ("last_traceback", "last_value", "last_exc"):
        if hasattr(sys, name):
            setattr(sys, name, None)
    for name in ("model", "trainer", "grpo_model"):
        globals().pop(name, None)
    gc.collect()
    torch.cuda.empty_cache()
    tok = load_tokenizer(INSTRUCT_MODEL_ID)
    tok.padding_side = "left"
    m = load_saved_model(INSTRUCT_MODEL_ID, cfg.model_id, torch.bfloat16)
    m.eval()
    partial = directory / "Instruct-Think.partial.json"
    details = json.loads(partial.read_text(encoding="utf-8")) if partial.exists() else []
    assert [r["index"] for r in details] == list(range(len(details)))
    eos = m.generation_config.eos_token_id
    eos_ids = eos if isinstance(eos, list) else [eos]
    started = time.monotonic()
    try:
        with tqdm(total=100, initial=len(details), desc="Instruct-Think batch", unit="题") as bar:
            while len(details) < 100:
                start = len(details)
                qs = questions[start:start + batch_size]
                prompts = [make_chat_prompt(tok, q, enable_thinking=True) for q in qs]
                inputs = tok(prompts, padding=True, return_tensors="pt").to(m.device)
                try:
                    with torch.inference_mode():
                        output = m.generate(**inputs, do_sample=False, max_new_tokens=2048,
                                            pad_token_id=tok.pad_token_id, use_cache=True)
                except torch.cuda.OutOfMemoryError:
                    if batch_size == 1:
                        raise
                    batch_size = max(1, batch_size // 2)
                    torch.cuda.empty_cache()
                    print("显存不足，降低 batch:", batch_size)
                    continue
                for offset, seq in enumerate(output[:, inputs["input_ids"].shape[1]:]):
                    ids = seq.tolist()
                    # 批生成时提前EOS的样本会补padding：长度统计只算到第一个EOS。
                    ending = next((j for j, token in enumerate(ids) if token in eos_ids), None)
                    actual = ids if ending is None else ids[:ending + 1]
                    text = tok.decode(actual, skip_special_tokens=True)
                    pred, parse_source = parse_eval_answer(text)
                    index = start + offset
                    gt = extract_gsm8k_answer(answers[index])
                    hit_limit = len(actual) >= 2048
                    details.append(dict(index=index, question=questions[index], response=text,
                                        ground_truth=gt, predicted=pred,
                                        correct=pred is not None and is_close(pred, gt),
                                        parse_source=parse_source,
                                        fallback_prediction=extract_answer_from_response(text),
                                        generated_tokens=len(actual), hit_limit=hit_limit,
                                        ended_with_eos=ending is not None,
                                        truncated=hit_limit and ending is None))
                # 原子写入，每完成一批即可恢复；旧前三个结果不改动。
                temp = partial.with_suffix(".tmp")
                temp.write_text(json.dumps(details, ensure_ascii=False, indent=2), encoding="utf-8")
                temp.replace(partial)
                bar.update(len(qs))
                del output, inputs
    finally:
        del m, tok
        gc.collect()
        torch.cuda.empty_cache()
    eligible = [r for r in details if not r["hit_limit"]]
    result = dict(name="Instruct-Think", total=100, correct=sum(r["correct"] for r in details),
                  accuracy=sum(r["correct"] for r in details)/100,
                  eligible_total=len(eligible), eligible_correct=sum(r["correct"] for r in eligible),
                  accuracy_non_limit=sum(r["correct"] for r in eligible)/len(eligible) if eligible else None,
                  hit_limit_count=sum(r["hit_limit"] for r in details),
                  truncated_count=sum(r["truncated"] for r in details),
                  parse_ok_count=sum(r["predicted"] is not None for r in details),
                  evaluation_batch_size=batch_size, resumed_seconds=time.monotonic()-started,
                  details=details)
    (directory / "Instruct-Think.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    results = [json.loads((directory / f"{name}.json").read_text(encoding="utf-8"))
               for name in ("Base", "GRPO", "Distill", "Instruct-Think")]
    fields = ["name", "correct", "total", "accuracy", "eligible_correct", "eligible_total",
              "accuracy_non_limit", "hit_limit_count", "truncated_count", "parse_ok_count"]
    with (directory / "summary.csv").open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows({key: r[key] for key in fields} for r in results)
    (directory / "instruct_batch_protocol.json").write_text(json.dumps({
        "model": INSTRUCT_MODEL_ID, "test_indices": list(range(100)), "do_sample": False,
        "max_new_tokens": 2048, "dtype": "bfloat16", "initial_batch_size": 4,
        "parser": "strict_v2", "note": "前三模型沿用逐题结果；批量推理可能存在数值差异"
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    for r in results:
        filtered = "N/A" if r["accuracy_non_limit"] is None else f"{100*r['accuracy_non_limit']:.1f}%"
        print(f"{r['name']}: 未触顶 {r['eligible_correct']}/{r['eligible_total']} ({filtered}); "
              f"全题 {r['correct']}/{r['total']}; 排除 {r['hit_limit_count']}")
    print("保存目录:", directory)
    return results


results = evaluate_instruct_batches(batch_size=4)
